# U2-02 · Salud, crédito y ventas: medidas y modelos lineales

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 2, apartados 2.1 a 2.3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar scikit-learn.

> Datos (UCI, licencia CC BY 4.0), junto a este cuaderno; su origen se explica en `LEEME.md`:
> Janosi, A., Steinbrunn, W., Pfisterer, M. y Detrano, R. (1989). *Heart Disease*. https://doi.org/10.24432/C52P4X ·
> Hofmann, H. (1994). *Statlog (German Credit Data)*. https://doi.org/10.24432/C5NC77 ·
> Sakar, C. O. y Kastro, Y. (2018). *Online Shoppers Purchasing Intention Dataset*. https://doi.org/10.24432/C5F88Q ·
> Chen, D. (2012). *Online Retail II*. https://doi.org/10.24432/C5CG6D

## 1. Caso de salud: enfermedad coronaria (2.1)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

d = pd.read_csv("corazon.csv")          # 303 pacientes de Cleveland
y = (d.pop("diagnostico") > 0).astype(int)   # 1 = enfermo
categoricas = ["sexo", "dolor", "glucosa", "ecg", "angina_esf",
               "pendiente", "talio"]
numericas = ["edad", "presion", "colesterol", "frec_max", "st_dep",
             "vasos"]

def preparacion():
    """Rellena los datos que faltan, codifica y escala."""
    cat = make_pipeline(SimpleImputer(strategy="most_frequent"),
                        OneHotEncoder(handle_unknown="ignore"))
    num = make_pipeline(SimpleImputer(strategy="median"),
                        StandardScaler())
    return make_column_transformer((cat, categoricas),
                                   (num, numericas))

X_ent, X_pru, y_ent, y_pru = train_test_split(
    d, y, test_size=0.3, stratify=y, random_state=0)

if __name__ == "__main__":
    print(d.shape, f"enfermos {y.mean():.1%}, datos que faltan "
          f"{d.isna().sum().sum()}")
    for nombre, m in [("referencia (clase mayoritaria)",
                       DummyClassifier()),
                      ("regresión logística", LogisticRegression())]:
        modelo = make_pipeline(preparacion(), m).fit(X_ent, y_ent)
        print(f"{nombre}: acierto en prueba "
              f"{modelo.score(X_pru, y_pru):.3f}")

## 2. Medidas de rendimiento y umbral (2.2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_score, StratifiedKFold

modelo = make_pipeline(preparacion(), LogisticRegression())
modelo.fit(X_ent, y_ent)
p = modelo.predict_proba(X_pru)[:, 1]        # probabilidad de enfermo

def medidas(umbral):
    vn, fp, fn, vp = confusion_matrix(y_pru, p >= umbral).ravel()
    return vp, fn, fp, vn, vp / (vp + fn), vn / (vn + fp)

for u in [0.5, 0.2]:
    vp, fn, fp, vn, sens, esp = medidas(u)
    print(f"umbral {u}: VP {vp} FN {fn} FP {fp} VN {vn} | "
          f"sensibilidad {sens:.3f}, especificidad {esp:.3f}")
print(f"área bajo la curva ROC: {roc_auc_score(y_pru, p):.3f}")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
acc = cross_val_score(modelo, d, y, cv=cv)     # 5 particiones
print("validación cruzada:", acc.round(3),
      f"media {acc.mean():.3f} ± {acc.std():.3f}")

fpr, tpr, umbrales = roc_curve(y_pru, p)
plt.plot(fpr, tpr, label="regresión logística")
plt.plot([0, 1], [0, 1], "--", lw=0.8, label="al azar")
for u in [0.5, 0.2]:
    _, _, _, _, s, e = medidas(u)
    plt.plot(1 - e, s, "o", ms=4)
    plt.annotate(f"umbral {u}", (1 - e, s), (6, -10),
                 textcoords="offset points")
plt.xlabel("1 - especificidad"); plt.ylabel("sensibilidad")
plt.legend(); plt.show()  # plt.savefig("roc.pdf")

**Prueba.** Busca el umbral más alto que mantenga la sensibilidad por encima de 0,95. ¿Cuántos sanos reciben una falsa alarma?

## 3. Caso de economía: crédito y matriz de costes (2.3)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

d = pd.read_csv("credito.csv")                 # 1000 clientes
y = (d.pop("riesgo") == 2).astype(int)         # 1 = mal pagador
cat = d.select_dtypes(exclude="number").columns  # A11, A12...
num = d.columns.difference(cat)
prep = make_column_transformer((OneHotEncoder(drop="first"), cat),
                               (StandardScaler(), num),
                               verbose_feature_names_out=False)
X_ent, X_pru, y_ent, y_pru = train_test_split(
    d, y, test_size=0.3, stratify=y, random_state=0)

# regularización L2: la validación cruzada elige su intensidad C
rejilla = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}
busca = GridSearchCV(make_pipeline(prep, LogisticRegression()),
                     rejilla, cv=5, scoring="roc_auc")
busca.fit(X_ent, y_ent)
modelo, lr = busca.best_estimator_, busca.best_estimator_[-1]
print(f"C elegido: {lr.C}; malos pagadores {y.mean():.0%}")
nombres = modelo[0].get_feature_names_out()
coef = pd.Series(lr.coef_[0], index=nombres).sort_values()
print("multiplican el riesgo (odds):")
print(np.exp(coef).round(2).iloc[[0, 1, 2, -3, -2, -1]])

# coste: aceptar a un mal pagador cuesta 5; rechazar a uno bueno, 1
p = modelo.predict_proba(X_pru)[:, 1]
def coste(rechazar):
    return 5 * np.sum(~rechazar & (y_pru == 1)) + \
           np.sum(rechazar & (y_pru == 0))
for nombre, r in [("aceptar a todos", np.zeros_like(p, bool)),
                  ("umbral 0.5", p >= 0.5),
                  ("umbral 1/6", p >= 1 / 6)]:
    print(f"{nombre:<16} coste {coste(r):4d}, rechaza {r.mean():.0%}")

**Prueba.** Quita las columnas `estado_sexo` y `extranjero` antes de entrenar (`d = d.drop(columns=[...])`). ¿Cambia mucho el coste?

## 4. Caso de empresa: previsión de ventas con un modelo lineal (2.3)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, r2_score,
                             root_mean_squared_error)

v = pd.read_csv("ventas_diarias.csv", parse_dates=["fecha"],
                index_col="fecha")["ventas"] / 1000  # miles de £

def variables(v):
    """Lo que se sabe la víspera: ventas pasadas y el calendario."""
    X = pd.DataFrame({"ayer": v.shift(1), "hace_7": v.shift(7),
                      "media_7": v.shift(1).rolling(7).mean(),
                      "media_28": v.shift(1).rolling(28).mean()})
    dias = pd.get_dummies(v.index.dayofweek, prefix="dia",
                          dtype=float).set_index(v.index)
    return X.join(dias).dropna()

X = variables(v)
y = v.loc[X.index]
ent, pru = X.index < "2011-07-01", X.index >= "2011-07-01"
print(f"entrenamiento {ent.sum()} días, prueba {pru.sum()} días")

def mide(nombre, pred):
    yp = y[pru]
    print(f"{nombre:<26} EAM {mean_absolute_error(yp, pred):5.2f}  "
          f"RECM {root_mean_squared_error(yp, pred):5.2f}  "
          f"R2 {r2_score(yp, pred):5.2f}")

if __name__ == "__main__":
    mide("referencia: hace 7 días", X.loc[pru, "hace_7"])
    lineal = LinearRegression().fit(X[ent], y[ent])
    mide("regresión lineal", lineal.predict(X[pru]))

---
## Listados de ampliación

Los listados siguientes completan los apartados del libro, en el mismo orden. Los que reutilizan un caso anterior lo importan de los ficheros `.py` que están junto a este cuaderno (por ejemplo, `from h_ventas import X, y`).

### Datos que faltan  ·  `e2_imputar.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from e_corazon import d, y, categoricas, numericas

falta = d.isna().any(axis=1)
print("faltan datos en:", d.columns[d.isna().any()].tolist(),
      "| pacientes afectados:", falta.sum())
print("enfermos entre ellos:", f"{y[falta].mean():.0%}",
      "| en el resto:", f"{y[~falta].mean():.0%}")

def con(imputador_num, indicador=False):
    cat = make_pipeline(SimpleImputer(strategy="most_frequent"),
                        OneHotEncoder(handle_unknown="ignore"))
    num = make_pipeline(imputador_num, StandardScaler())
    if indicador:                 # añade una columna «faltaba»
        num.steps[0][1].set_params(add_indicator=True)
    return make_pipeline(make_column_transformer(
        (cat, categoricas), (num, numericas)), LogisticRegression())

cv = StratifiedKFold(10, shuffle=True, random_state=0)
for nombre, m, X, yy in [
        ("quitar pacientes", con(SimpleImputer()), d[~falta],
         y[~falta]),
        ("mediana", con(SimpleImputer(strategy="median")), d, y),
        ("mediana + indicador",
         con(SimpleImputer(strategy="median"), True), d, y),
        ("5 vecinos", con(KNNImputer(n_neighbors=5)), d, y)]:
    acc = cross_val_score(m, X, yy, cv=cv)
    print(f"{nombre:<20} n={len(yy)} acierto {acc.mean():.3f}"
          f" ± {acc.std():.3f}")

### Fugas de información en la preparación  ·  `e5_fuga.py`

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(0)
X = rng.normal(size=(200, 5000))     # 5000 variables de puro ruido
y = rng.integers(0, 2, 200)          # etiquetas al azar
# MAL: elegir las 20 «mejores» variables con todos los datos
X20 = SelectKBest(f_classif, k=20).fit_transform(X, y)
mal = cross_val_score(LogisticRegression(), X20, y, cv=5)
# BIEN: la selección dentro del pipeline, en cada partición
bien = cross_val_score(make_pipeline(
    SelectKBest(f_classif, k=20), LogisticRegression()), X, y, cv=5)
print(f"selección fuera de la validación: acierto {mal.mean():.3f}")
print(f"selección dentro del pipeline:    acierto {bien.mean():.3f}")

### Codificar las variables categóricas  ·  `e6_codificar.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder,
                                   StandardScaler)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

d = pd.read_csv("credito.csv")
y = (d.pop("riesgo") == 2).astype(int)
cat = d.select_dtypes(exclude="number").columns
num = d.columns.difference(cat)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for nombre_cod, cod in [("one-hot", OneHotEncoder(drop="first")),
                        ("ordinal", OrdinalEncoder())]:
    prep = make_column_transformer((cod, cat),
                                   (StandardScaler(), num))
    n = prep.fit_transform(d).shape[1]
    for nombre, m in [("logística", LogisticRegression(
                          max_iter=2000)),
                      ("potenciación", HistGradientBoostingClassifier(
                          max_iter=100, learning_rate=0.05,
                          random_state=0))]:
        auc = cross_val_score(make_pipeline(prep, m), d, y, cv=cv,
                              scoring="roc_auc").mean()
        print(f"{nombre_cod:<8} ({n:2d} columnas) {nombre:<13}"
              f" AUC {auc:.3f}")

### Más de dos clases  ·  `e3_multiclase.py`

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report

m = sm.datasets.modechoice.load_pandas().data   # 210 viajeros x 4
modos = {1: "avión", 2: "tren", 3: "autobús", 4: "coche"}
m["modo"] = m["mode"].map(modos)
ancho = m.pivot(index="individual", columns="modo",
                values=["ttme", "gc"])        # una fila por viajero
ancho.columns = [f"{v}_{k}" for v, k in ancho.columns]
datos = m.groupby("individual")[["hinc", "psize"]].first()
X = ancho.join(datos)                         # 10 variables
y = m[m.choice == 1].set_index("individual")["modo"].loc[X.index]
print(", ".join(f"{k} {v}" for k, v in y.value_counts().items()))

modelo = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=1000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
pred = cross_val_predict(modelo, X, y, cv=cv)
orden = list(modos.values())
print(pd.DataFrame(confusion_matrix(y, pred, labels=orden),
                   index=orden, columns=orden))
print(classification_report(y, pred, digits=3,
                            zero_division=0))

### Más de dos clases  ·  `e4_gravedad.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, balanced_accuracy_score
from e_corazon import d, preparacion

g = pd.read_csv("corazon.csv")["diagnostico"]   # 0 a 4
print("pacientes por grado:", g.value_counts().sort_index().to_dict())
modelo = make_pipeline(preparacion(),
                       LogisticRegression(max_iter=2000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
p = cross_val_predict(modelo, d, g, cv=cv)
print(confusion_matrix(g, p))
print(f"acierto {(p == g).mean():.3f}, equilibrado "
      f"{balanced_accuracy_score(g, p):.3f}")
# ¿y si solo importa enfermo o sano?
print(f"acierto enfermo/sano con este modelo: "
      f"{((p > 0) == (g > 0)).mean():.3f}")

### La incertidumbre de una medida  ·  `f4_bootstrap.py`

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from e_corazon import X_ent, X_pru, y_ent, y_pru, preparacion

modelo = make_pipeline(preparacion(), LogisticRegression())
p = modelo.fit(X_ent, y_ent).predict_proba(X_pru)[:, 1]
y = y_pru.to_numpy()
rng = np.random.default_rng(0)
aucs = []
for _ in range(2000):                  # remuestreos de la prueba
    i = rng.integers(0, len(y), len(y))
    if y[i].min() != y[i].max():       # hacen falta las dos clases
        aucs.append(roc_auc_score(y[i], p[i]))
bajo, alto = np.percentile(aucs, [2.5, 97.5])
print(f"{len(y)} pacientes de prueba, AUC {roc_auc_score(y, p):.3f}")
print(f"intervalo bootstrap del 95 %: {bajo:.3f} a {alto:.3f}")

### La incertidumbre de una medida  ·  `f5_comparar.py`

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (RepeatedStratifiedKFold,
                                     cross_val_score)
from e_corazon import d, y, preparacion

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10,
                             random_state=0)
a = cross_val_score(make_pipeline(preparacion(),
                    LogisticRegression()), d, y, cv=cv,
                    scoring="roc_auc")
b = cross_val_score(make_pipeline(preparacion(),
                    RandomForestClassifier(300, random_state=0)),
                    d, y, cv=cv, scoring="roc_auc")
dif = a - b              # mismas particiones: diferencias pareadas
print(f"logística {a.mean():.3f}, bosque {b.mean():.3f}")
print(f"diferencia media {dif.mean():.3f}, gana la logística en "
      f"{(dif > 0).sum()} de {len(dif)} particiones")
print(f"desviación de las diferencias {dif.std():.3f}")

### Probabilidades en las que se puede confiar  ·  `f2_calibrar.py`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import (CalibratedClassifierCV,
                                 calibration_curve)
from sklearn.metrics import brier_score_loss, roc_auc_score
from i_compras import prep, X_ent, X_pru, y_ent, y_pru

modelos = {
    "logística": make_pipeline(prep(True),
                               LogisticRegression(max_iter=1000)),
    "Bayes ingenuo": make_pipeline(prep(True), GaussianNB()),
    "Bayes calibrado": CalibratedClassifierCV(
        make_pipeline(prep(True), GaussianNB()),
        method="isotonic", cv=5),
    "bosque aleatorio": make_pipeline(prep(False),
        RandomForestClassifier(300, n_jobs=-1, random_state=0))}
fig, eje = plt.subplots(figsize=(3.2, 2.6))
for nombre, m in modelos.items():
    p = m.fit(X_ent, y_ent).predict_proba(X_pru)[:, 1]
    print(f"{nombre:<17} Brier {brier_score_loss(y_pru, p):.3f}"
          f"  AUC {roc_auc_score(y_pru, p):.3f}"
          f"  media prevista {p.mean():.3f}")
    real, prev = calibration_curve(y_pru, p, n_bins=10,
                                   strategy="quantile")
    eje.plot(prev, real, "o-", ms=2.5, label=nombre)
print(f"proporción real de compras en prueba: {y_pru.mean():.3f}")
eje.plot([0, 1], [0, 1], "--", c="0.6", lw=0.8)
eje.set_xlabel("probabilidad prevista")
eje.set_ylabel("proporción observada"); eje.legend(fontsize=6)
plt.show()  # fig.savefig("calibrar.pdf")

### Clases desequilibradas  ·  `k3_desequilibrio.py`

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score,
                             average_precision_score, roc_auc_score)
from i_compras import prep, X_ent, X_pru, y_ent, y_pru

print(f"compras en prueba: {y_pru.sum()} de {len(y_pru)}")
for pesos in [None, "balanced", "balanced_subsample"]:
    m = make_pipeline(prep(False), RandomForestClassifier(
        300, min_samples_leaf=3, class_weight=pesos, n_jobs=-1,
        random_state=0)).fit(X_ent, y_ent)
    p, c = m.predict_proba(X_pru)[:, 1], m.predict(X_pru)
    print(f"pesos {str(pesos):<18} precisión "
          f"{precision_score(y_pru, c):.3f}  sensibilidad "
          f"{recall_score(y_pru, c):.3f}")
    print(f"{'':<24} AUC-ROC {roc_auc_score(y_pru, p):.3f}  "
          f"AUC-PR {average_precision_score(y_pru, p):.3f}")
print(f"AUC-PR de un modelo al azar: {y_pru.mean():.3f}")

### Elegir el umbral con los datos  ·  `f6_umbral.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve, f1_score
from i_compras import prep, X_ent, X_pru, y_ent, y_pru

m = make_pipeline(prep(False),
                  HistGradientBoostingClassifier(random_state=0))
# el umbral se elige con validación cruzada en entrenamiento
p_cv = cross_val_predict(m, X_ent, y_ent, cv=5,
                         method="predict_proba")[:, 1]
prec, sens, umb = precision_recall_curve(y_ent, p_cv)
f1 = 2 * prec * sens / (prec + sens + 1e-12)
mejor = umb[np.argmax(f1[:-1])]
print(f"umbral con mejor F1 en validación: {mejor:.2f}")
p = m.fit(X_ent, y_ent).predict_proba(X_pru)[:, 1]
for u in [0.5, mejor]:
    print(f"umbral {u:.2f}: F1 en prueba "
          f"{f1_score(y_pru, p >= u):.3f}, avisos {(p >= u).sum()}")
fig, eje = plt.subplots(figsize=(4.4, 1.9))
eje.plot(umb, prec[:-1], label="precisión")
eje.plot(umb, sens[:-1], label="sensibilidad")
eje.plot(umb, f1[:-1], label="F1")
eje.axvline(mejor, ls="--", c="0.5", lw=0.8)
eje.set_xlabel("umbral"); eje.legend(); plt.show()  # fig.savefig("umbral.pdf")

### Curva de ganancia: a quién dirigirse  ·  `f7_ganancia.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from i_compras import prep, X_ent, X_pru, y_ent, y_pru

m = make_pipeline(prep(False),
                  HistGradientBoostingClassifier(random_state=0))
p = m.fit(X_ent, y_ent).predict_proba(X_pru)[:, 1]
orden = np.argsort(-p)                       # de más a menos probable
captadas = np.cumsum(y_pru.to_numpy()[orden]) / y_pru.sum()
frac = np.arange(1, len(p) + 1) / len(p)
for f in [0.05, 0.10, 0.20, 0.30]:
    k = int(f * len(p))
    print(f"contactar al {f:3.0%}: {captadas[k - 1]:.0%} de las "
          f"compras (lift {captadas[k - 1] / f:.1f})")
fig, eje = plt.subplots(figsize=(4.4, 2.2))
eje.plot(frac, captadas, label="modelo")
eje.plot([0, 1], [0, 1], "--", c="0.6", lw=0.8, label="al azar")
eje.plot([0, y_pru.mean(), 1], [0, 1, 1], ":", c="0.3", lw=0.8,
         label="perfecto")
eje.set_xlabel("proporción de visitas contactadas")
eje.set_ylabel("proporción de compras"); eje.legend()
plt.show()  # fig.savefig("ganancia.pdf")

### La regresión logística de statsmodels  ·  `g4_logit.py`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

d = pd.read_csv("credito.csv")
d["malo"] = (d.riesgo == 2).astype(int)
d["importe_miles"] = d.importe / 1000
m = smf.logit("malo ~ duracion + importe_miles + edad + C(cuenta)",
              data=d).fit(disp=0)
tabla = pd.DataFrame({"coef": m.params, "p": m.pvalues,
                      "odds": np.exp(m.params)}).round(3)
print(tabla.to_string())
print(f"pseudo-R2 de McFadden: {m.prsquared:.3f}")
ef = m.get_margeff(at="mean").summary_frame()
print("efecto marginal en la media (puntos de probabilidad):")
print((100 * ef["dy/dx"]).round(2).iloc[-3:].to_string())

### ¿Trata el modelo igual a todos?  ·  `g5_equidad.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold

d = pd.read_csv("credito.csv")
y = (d.pop("riesgo") == 2).astype(int)
grupo = pd.DataFrame({
    "extranjero": d.extranjero.map({"A201": "sí", "A202": "no"}),
    "mujer": d.estado_sexo.map(lambda v: "sí" if v == "A92"
                               else "no")})
filas = {}
for quitar in [[], ["estado_sexo", "extranjero"]]:
    X = d.drop(columns=quitar)
    cat = X.select_dtypes(exclude="number").columns
    prep = make_column_transformer(
        (OneHotEncoder(drop="first"), cat),
        (StandardScaler(), X.columns.difference(cat)))
    p = cross_val_predict(make_pipeline(prep, LogisticRegression(
        max_iter=1000)), X, y, method="predict_proba",
        cv=StratifiedKFold(5, shuffle=True, random_state=0))[:, 1]
    rechazo = pd.Series(p >= 1 / 6, index=d.index)
    filas["rechazo" + (" sin" if quitar else "")] = pd.concat(
        [rechazo.groupby(grupo[g]).mean() for g in grupo],
        keys=grupo.columns)
tabla = pd.DataFrame(filas)
tabla.insert(0, "n", pd.concat([grupo[g].value_counts()
                                for g in grupo], keys=grupo.columns))
tabla.insert(1, "malos", pd.concat([y.groupby(grupo[g]).mean()
                                    for g in grupo],
                                   keys=grupo.columns))
tabla.index.names = [None, None]
print(tabla.round(2).to_string())

### Relaciones no lineales con splines  ·  `g6_splines.py`

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import (OneHotEncoder, StandardScaler,
                                   SplineTransformer)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

d = pd.read_csv("credito.csv")
y = (d.pop("riesgo") == 2).astype(int)
cat = d.select_dtypes(exclude="number").columns
otras = d.columns.difference(cat).difference(["edad"])
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for nombre, edad in [
        ("edad lineal", StandardScaler()),
        ("edad con splines", SplineTransformer(n_knots=5, degree=3))]:
    prep = make_column_transformer(
        (OneHotEncoder(drop="first"), cat),
        (StandardScaler(), otras), (edad, ["edad"]))
    m = make_pipeline(prep, LogisticRegression(max_iter=2000))
    auc = cross_val_score(m, d, y, cv=cv, scoring="roc_auc")
    print(f"{nombre:<17} AUC {auc.mean():.3f} ± {auc.std():.3f}")
# riesgo observado por tramos de edad
tramos = pd.cut(d.edad, [18, 25, 30, 35, 45, 55, 76],
                labels=["19-25", "26-30", "31-35", "36-45",
                        "46-55", "56-75"])
print("% de malos por edad:", (y.groupby(tramos, observed=True)
      .mean() * 100).round(0).astype(int).to_dict())

### Validación cruzada con series temporales  ·  `f3_temporal.py`

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (KFold, TimeSeriesSplit,
                                     cross_val_score)
from h_ventas import X, y                     # el caso de 2.3

modelo = LinearRegression()
esquemas = {
    "5 partes barajadas": KFold(5, shuffle=True, random_state=0),
    "5 partes en orden": TimeSeriesSplit(5),
    "en orden, hueco de 7 días": TimeSeriesSplit(5, gap=7)}
for nombre, cv in esquemas.items():
    eam = -cross_val_score(modelo, X, y, cv=cv,
                           scoring="neg_mean_absolute_error")
    print(f"{nombre:<26} EAM por parte {eam.round(1)}"
          f" media {eam.mean():.2f}")
for i, (e, p) in enumerate(TimeSeriesSplit(5).split(X)):
    print(f"parte {i + 1}: entrena {X.index[e[0]]:%d/%m/%Y}-"
          f"{X.index[e[-1]]:%d/%m/%Y}, prueba hasta "
          f"{X.index[p[-1]]:%d/%m/%Y}")

### Descomponer la serie  ·  `h4_descomponer.py`

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from h_ventas import v                         # miles de £ al día

d = STL(v, period=7, robust=True).fit()        # semana de 7 días
total = v.var()
print(f"varianza sobre la total: tendencia "
      f"{d.trend.var() / total:.0%},"
      f" semanal {d.seasonal.var() / total:.0%},"
      f" resto {d.resid.var() / total:.0%}")
semana = d.seasonal.groupby(v.index.dayofweek).mean()
print("efecto medio de cada día:",
      dict(zip("LMXJVSD", semana.round(1).tolist())))
print("tendencia: dic 2010", round(d.trend["2010-12"].mean(), 1),
      "| dic 2011", round(d.trend["2011-12"].mean(), 1))
fig, ejes = plt.subplots(3, 1, figsize=(4.4, 3.2), sharex=True,
                         layout="constrained")
for eje, s, t in zip(ejes, [d.trend, d.seasonal, d.resid],
                     ["tendencia", "ciclo semanal", "resto"]):
    eje.plot(s, lw=0.6); eje.set_ylabel(t)
fig.autofmt_xdate(); plt.show()  # fig.savefig("stl.pdf")

### Un modelo clásico de series: SARIMA  ·  `h2_sarimax.py`

In [ ]:
import warnings
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
from h_ventas import v, y, pru                 # el caso de 2.3

warnings.simplefilter("ignore")        # avisos de convergencia
serie = v.asfreq("D", fill_value=0)            # todos los días
ent = serie[serie.index < "2011-07-01"]
modelo = SARIMAX(ent, order=(1, 0, 1), seasonal_order=(1, 0, 1, 7),
                 trend="c").fit(disp=False)
print(f"AIC {modelo.aic:.0f}")
# un paso: cada día se predice con los datos hasta la víspera
completo = modelo.apply(serie)                 # mismos parámetros
pred = completo.predict(start="2011-07-01")
fechas = y[pru].index
print(f"SARIMA un día vista: EAM "
      f"{mean_absolute_error(y[pru], pred[fechas]):.2f}")
lejos = modelo.forecast(len(serie) - len(ent))  # todo de una vez
print(f"SARIMA sin datos nuevos: EAM "
      f"{mean_absolute_error(y[pru], lejos[fechas]):.2f}")

### Prever con más antelación  ·  `h3_horizonte.py`

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from h_ventas import v                          # miles de £ al día

def variables(v, h):
    """Lo que se sabe h días antes del día que se predice."""
    X = pd.DataFrame({"ultimo": v.shift(h),
                      "semana": v.shift(7 * ((h + 6) // 7)),
                      "media_28": v.shift(h).rolling(28).mean()})
    X["dia"] = v.index.dayofweek
    X["mes"] = v.index.month
    return X.dropna()

for h in [1, 7, 14, 28]:
    X = variables(v, h)
    y = v.loc[X.index]
    ent, pru = X.index < "2011-07-01", X.index >= "2011-07-01"
    m = RandomForestRegressor(300, min_samples_leaf=3,
                              random_state=0, n_jobs=-1)
    m.fit(X[ent], y[ent])
    eam = mean_absolute_error(y[pru], m.predict(X[pru]))
    print(f"a {h:>2} días vista: EAM {eam:5.2f} miles de £")

### Regularización: ridge y lasso  ·  `g2_lasso.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from h_ventas import v                         # miles de £ al día

X = pd.DataFrame({f"r{k}": v.shift(k) for k in range(1, 29)})
X = X.join(pd.get_dummies(v.index.dayofweek, prefix="dia",
                          drop_first=True, dtype=float)
           .set_index(v.index)).dropna()       # 28 retardos + 6 días
y = v.loc[X.index]
ent, pru = X.index < "2011-07-01", X.index >= "2011-07-01"
cv = TimeSeriesSplit(5)
modelos = {"MCO, 34 variables": LinearRegression(),
           "ridge": RidgeCV(alphas=np.logspace(-2, 4, 30)),
           "lasso": LassoCV(cv=cv, random_state=0, max_iter=50000)}
for nombre, m in modelos.items():
    m = make_pipeline(StandardScaler(), m).fit(X[ent], y[ent])
    eam = mean_absolute_error(y[pru], m.predict(X[pru]))
    coef = pd.Series(m[-1].coef_, X.columns)
    print(f"{nombre:<18} EAM prueba {eam:5.2f}, coeficientes"
          f" no nulos {(coef.abs() > 1e-6).sum()}")
print("lasso, alfa elegido:", round(m[-1].alpha_, 3))
quita = [c for c in coef.index[:28] if abs(coef[c]) <= 1e-6]
print("retardos que anula:", ", ".join(quita))
coef.iloc[:28].plot.bar(figsize=(4.4, 1.8), width=0.8)
plt.xticks(fontsize=5); plt.ylabel("coeficiente (escalado)")
plt.show()  # plt.savefig("lasso.pdf")

### Contar sucesos: regresión de Poisson  ·  `g3_poisson.py`

In [ ]:
import warnings
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

r = sm.datasets.randhie.load_pandas().data     # 20 190 personas
print(f"visitas al médico: media {r.mdvis.mean():.2f}, varianza "
      f"{r.mdvis.var():.2f}, ceros {(r.mdvis == 0).mean():.0%}")
formula = ("mdvis ~ lncoins + idp + lpi + fmde + physlm + disea"
           " + hlthg + hlthf + hlthp")
pois = smf.glm(formula, r, family=sm.families.Poisson()).fit()
print("Poisson: razones de tasas (exp de los coeficientes)")
print(np.exp(pois.params).round(3).iloc[1:].to_string())
dispersion = pois.pearson_chi2 / pois.df_resid
print(f"dispersión (debería ser 1): {dispersion:.2f}")
warnings.simplefilter("ignore")          # avisos de convergencia
nb = smf.negativebinomial(formula, r).fit(disp=0)
print(f"AIC Poisson {pois.aic:.0f}  binomial negativa {nb.aic:.0f}")
print("efecto del copago con la binomial negativa:",
      round(np.exp(nb.params["lncoins"]), 3),
      "IC 95 %", np.exp(nb.conf_int().loc["lncoins"]).round(3)
      .tolist())

**Ejercicio.** Añade la variable `hace_14` (las ventas de hace dos semanas). ¿Mejora la regresión lineal?

> Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, 27(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010